# 182070 carrier: four-corner adapter geometry test

Inspect the **bare 182531 adapters** on the **182070 carrier at rail 13**. No plates are installed.
A single **50 µL filtered tip** comes from **TIP_CAR_480_A00, position [2], A1**, at **rail 25**.
Channel **0** (physical channel 1) visits **module[0] H1 → H12 → module[4] A1 → A12**.

Run cells from top to bottom in a dedicated kernel. The configuration and geometry preview cells
do not connect to the robot. Setup initializes hardware; verify the deck and clear the movement
path before running it. Do not rerun configuration while connected.
After PyLabRobot dimensions change, finish any tip recovery and disconnect, then restart
the kernel and rerun from the top to load the revised definitions and rebuild the deck.

At every corner, press **Enter** after inspecting the tip 5 mm above the opening to descend to
the modeled adapter top; press **Enter** again after inspecting the top to raise and continue.
Enter **q** (or any nonempty response) to cancel, or interrupt the cell. Cancellation attempts
only a vertical raise and leaves tip return to the recovery cell. No liquid is aspirated or dispensed.

In [ ]:
from pathlib import Path
import math
import sys

# Use the recently updated local resource definitions.
PYLABROBOT_ROOT = Path("/home/hamilton-robot/Documents/Hamilton-Starlet/pylabrobot")
if not (PYLABROBOT_ROOT / "pylabrobot" / "__init__.py").is_file():
    raise RuntimeError(f"PyLabRobot checkout not found: {PYLABROBOT_ROOT}")
if str(PYLABROBOT_ROOT) not in sys.path:
    sys.path.insert(0, str(PYLABROBOT_ROOT))

import pylabrobot

if Path(pylabrobot.__file__).resolve().parent != (PYLABROBOT_ROOT / "pylabrobot").resolve():
    raise RuntimeError("Another PyLabRobot checkout is already loaded. Restart this kernel.")

from pylabrobot.legacy.liquid_handling import LiquidHandler
from pylabrobot.legacy.liquid_handling.backends import STARBackend
from pylabrobot.resources import (
    Coordinate,
    PlateAdapter,
    PLT_CAR_L5PCR,
    STARLetDeck,
    TIP_CAR_480_A00,
    hamilton_96_tiprack_50uL_filter,
)
from pylabrobot.resources.tip_tracking import set_tip_tracking

if "lh" in globals() and (lh.setup_finished or globals().get("needs_recovery", False)):
    raise RuntimeError("Keep the existing handler. Recover the tip and stop before reconfiguring.")

# Change these only before constructing the deck / connecting.
CARRIER_RAIL = 13
TIP_CARRIER_RAIL = 25
CHANNEL = 0  # zero-based: physical channel 1
TIP_CARRIER_POSITION = 2  # zero-based: third rack
TIP_POSITION = "A1"
TEST_CORNERS = ((0, "H1"), (0, "H12"), (4, "A1"), (4, "A12"))
SAFE_TRAVERSE_Z = 247.6  # absolute tip-end Z, mm
ABOVE_TOP_CLEARANCE_MM = 5.0

set_tip_tracking(True)
backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())

carrier = PLT_CAR_L5PCR("carrier_182070", with_adapters=True)
lh.deck.assign_child_resource(carrier, track=CARRIER_RAIL)
tip_carrier = TIP_CAR_480_A00("tip_carrier")
tip_rack = hamilton_96_tiprack_50uL_filter("tips_50ul_filter")
tip_carrier[TIP_CARRIER_POSITION] = tip_rack
lh.deck.assign_child_resource(tip_carrier, track=TIP_CARRIER_RAIL)
tip_spot = tip_rack.get_item(TIP_POSITION)
probe_tip = tip_spot.get_tip()
needs_recovery = False

In [ ]:
def hole_top_center(adapter, hole):
    """Absolute center of a bare adapter opening; H is front, A is rear."""
    if hole[0] not in "ABCDEFGH" or not hole[1:].isdigit():
        raise ValueError(f"Invalid 96-well identifier: {hole!r}")
    column = int(hole[1:]) - 1
    if not 0 <= column < 12:
        raise ValueError(f"Invalid 96-well column: {hole!r}")
    row_from_front = 7 - "ABCDEFGH".index(hole[0])
    origin = adapter.get_absolute_location()
    return Coordinate(
        origin.x + adapter.dx + adapter.adapter_hole_size_x / 2
        + column * adapter.adapter_hole_dx,
        origin.y + adapter.dy + adapter.adapter_hole_size_y / 2
        + row_from_front * adapter.adapter_hole_dy,
        adapter.get_absolute_location(z="top").z,
    )


def geometry_targets():
    if list(carrier.sites) != list(range(5)):
        raise ValueError("The carrier must contain five sites, indexed 0 through 4.")
    for index, site in carrier.sites.items():
        adapter = site.resource
        if not isinstance(adapter, PlateAdapter) or adapter.model != "Hamilton_96_adapter_182531":
            raise ValueError(f"Module {index} must contain a Hamilton 182531 adapter.")
        if adapter.parent is not site or site.parent is not carrier or adapter.children:
            raise ValueError(f"Module {index} must be mounted on the carrier with no plate.")
        if adapter.get_absolute_rotation().z % 360 != 0:
            raise ValueError("This test expects unrotated landscape adapters.")
    if tip_spot.parent is not tip_rack or tip_carrier[TIP_CARRIER_POSITION].resource is not tip_rack:
        raise ValueError("Tip rack is not mounted in the configured tip-carrier position.")
    if probe_tip.nominal_volume != 50 or not probe_tip.has_filter:
        raise ValueError("This test requires a 50 µL filtered tip.")
    targets = [
        (index, hole, hole_top_center(carrier[index].resource, hole))
        for index, hole in TEST_CORNERS
    ]
    tip_overhang = probe_tip.get_size_z() - backend.DEFAULT_TIP_FITTING_DEPTH
    min_tip_z = backend.MINIMUM_CHANNEL_Z_POSITION - tip_overhang
    max_tip_z = backend.MAXIMUM_CHANNEL_Z_POSITION - tip_overhang
    for index, hole, point in targets:
        if not all(math.isfinite(v) for v in (point.x, point.y, point.z)):
            raise ValueError(f"Nonfinite target for module {index}, {hole}.")
        if not min_tip_z <= point.z < point.z + ABOVE_TOP_CLEARANCE_MM < SAFE_TRAVERSE_Z <= max_tip_z:
            raise ValueError("Top, approach, and traverse Z must fit the 50 µL tip's movement range.")
    return targets


targets = geometry_targets()
print(f"PyLabRobot: {pylabrobot.__file__}")
print(lh.deck.summary())
print(f"Tip: rail {TIP_CARRIER_RAIL}, position [{TIP_CARRIER_POSITION}], {TIP_POSITION}")
print(f"Channel: {CHANNEL} (physical {CHANNEL + 1}); traverse tip-end Z: {SAFE_TRAVERSE_Z:.2f} mm")
print("\nModule  Hole        X        Y    Top Z  Approach Z  (absolute mm)")
for index, hole, point in targets:
    print(f"{index:6d}  {hole:4s}  {point.x:7.2f}  {point.y:7.2f}  {point.z:7.2f}  "
          f"{point.z + ABOVE_TOP_CLEARANCE_MM:10.2f}")
print(f"Horizontal corner span: {targets[1][2].x - targets[0][2].x:.2f} mm")
print(f"Front-to-rear corner span: {targets[2][2].y - targets[0][2].y:.2f} mm")

With the Hamilton VENUS template and 182531 insert model, the preview should show:

| Module | Hole | X (mm) | Y (mm) | Top Z (mm) |
|---|---|---:|---:|---:|
| 0 | H1 | 387.50 | 83.60 | 220.70 |
| 0 | H12 | 486.50 | 83.60 | 220.70 |
| 4 | A1 | 387.50 | 530.60 | 220.70 |
| 4 | A12 | 486.50 | 530.60 | 220.70 |

The horizontal span is **99.00 mm**; the front-to-rear span is **447.00 mm**.
These are reference values, not motion constants. Targets use each adapter's geometry and top
surface, not its modeled cavity depth or plate seating offset. No calibration offsets are applied.

In [ ]:
class InspectionCancelled(RuntimeError):
    pass


def pause_for_inspection(message):
    response = input(f"\n{message}\nPress Enter to continue, or q to cancel: ")
    if response.strip():
        raise InspectionCancelled("Operator cancelled the geometry test.")


async def validate_before_pickup():
    if not lh.setup_finished:
        raise RuntimeError("Run the robot setup cell first.")
    if needs_recovery:
        raise RuntimeError("Run the recovery cell before starting another test.")
    current_targets = geometry_targets()
    if not isinstance(CHANNEL, int) or not 0 <= CHANNEL < backend.num_channels:
        raise ValueError(f"Channel {CHANNEL} is unavailable on this robot.")
    if not tip_spot.has_tip():
        raise RuntimeError("The source A1 spot has no modeled tip.")
    if not backend.can_pick_up_tip(CHANNEL, probe_tip):
        raise ValueError("The selected channel cannot pick up the specified tip.")
    arm = backend.extended_conf.left_x_drive
    if arm is None or arm.x_range is None:
        raise RuntimeError("The robot has not reported its left-arm X travel range.")
    x_min, x_max = arm.x_range
    positions = [("tip source", tip_spot.get_absolute_location(x="c", y="c"))]
    positions += [(f"module {i}, {hole}", point) for i, hole, point in current_targets]
    for label, point in positions:
        if not x_min <= point.x <= x_max or not backend.can_reach_position(CHANNEL, point):
            raise ValueError(f"Channel {CHANNEL} cannot reach {label} at {point}.")
    # Other channels share the X arm; this single-tip test starts with all channels empty.
    if any(await backend.request_tip_presence()) or any(t.has_tip for t in lh.head.values()):
        raise RuntimeError("Start with all pipetting channels empty.")
    return current_targets


async def run_geometry_test():
    global needs_recovery
    current_targets = await validate_before_pickup()
    pause_for_inspection(
        f"Ready to pick up the 50 µL tip at rail {TIP_CARRIER_RAIL}, "
        f"position [{TIP_CARRIER_POSITION}], {TIP_POSITION}, with channel {CHANNEL}."
    )
    # Pickup can fail after physically attaching a tip. Preserve recovery state in that case.
    needs_recovery = True
    try:
        await lh.pick_up_tips([tip_spot], use_channels=[CHANNEL])
        await backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
        await lh.prepare_for_manual_channel_operation(CHANNEL)

        for index, hole, point in current_targets:
            # All lateral movement occurs after raising the tip above the adapters.
            await backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
            await lh.move_channel_x(CHANNEL, point.x)
            await lh.move_channel_y(CHANNEL, point.y)
            await backend.move_channel_tool_z(
                channel_idx=CHANNEL, z=point.z + ABOVE_TOP_CLEARANCE_MM
            )
            pause_for_inspection(
                f"Module[{index}] {hole}: X={point.x:.2f}, Y={point.y:.2f}, "
                f"5 mm above the opening. Inspect alignment; Enter lowers to Z={point.z:.2f}."
            )
            await backend.move_channel_tool_z(channel_idx=CHANNEL, z=point.z)
            pause_for_inspection(
                f"Module[{index}] {hole}: tip at the modeled top, Z={point.z:.2f}. "
                "Inspect centering; Enter raises the tip and continues."
            )
            await backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)

        await lh.drop_tips([tip_spot], use_channels=[CHANNEL])
        needs_recovery = False
        print("All four corners inspected; tip returned to position [2], A1.")
    finally:
        if needs_recovery:
            print("Test interrupted. Attempting only a vertical raise; no automatic tip return.")
            try:
                await backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
            except Exception as error:
                print(f"Vertical raise did not complete: {error!r}")
            print("Inspect the robot, then use the recovery cell. Do not restart setup.")


async def recover_tip():
    global needs_recovery
    if not lh.setup_finished:
        raise RuntimeError("The handler must remain connected for recovery.")
    present = (await backend.request_tip_presence())[CHANNEL]
    modeled = lh.head[CHANNEL].has_tip
    if not present and not modeled and tip_spot.has_tip():
        needs_recovery = False
        print("No tip attached; the source A1 is still modeled as occupied. Verify it visually.")
        return
    if not present or not modeled or tip_spot.has_tip():
        raise RuntimeError(
            "Physical and modeled tip states disagree. Recover manually before further movement."
        )
    pause_for_inspection(
        "Confirm the return path is clear and the original position [2], A1 is empty. "
        "Enter raises the tip and returns it to that spot."
    )
    await backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
    await lh.drop_tips([tip_spot], use_channels=[CHANNEL])
    needs_recovery = False
    print("Tip returned to position [2], A1.")

## Connect and initialize

Confirm the preview matches the physical deck: carrier at rail 13, all five adapters mounted
without plates, tip carrier at rail 25, and a 50 µL filtered tip at position [2], A1.
Start with empty pipetting channels and a clear movement path. This cell initializes hardware;
the next cell performs the inspection sequence.

In [ ]:
if lh.setup_finished:
    print("Handler is already initialized; setup was not repeated.")
else:
    if needs_recovery:
        raise RuntimeError("Recover the previous run before initializing again.")
    pause_for_inspection("Deck checked and channels empty. Ready to initialize the robot.")
    await lh.setup(skip_autoload=True)

In [ ]:
await run_geometry_test()

## Recovery after interruption

Run this cell only when a test was interrupted. It checks physical tip presence against the
tracked state, raises vertically, and returns the tip to the original A1 spot after your
confirmation. If pickup or return failed and the states disagree, inspect and recover manually;
do not create a new handler or repeat setup to bypass the state check.

In [ ]:
await recover_tip()

## Disconnect

Run after successful completion or recovery. Disconnecting does not return a mounted tip.

In [ ]:
if needs_recovery:
    raise RuntimeError("Resolve tip recovery before disconnecting.")
if lh.setup_finished:
    await lh.stop()
    print("Disconnected.")
else:
    print("Handler is already disconnected.")